In [ ]:
from valdpy import ValdAuth, NordBordAPI
from valdpy.utils import read_credentials

import pandas as pd
from datetime import datetime

%load_ext autoreload
%autoreload 2

# NordBord API Example

This example demonstrates how to use the VALDPY package to access NordBord (leg press strength) test data.

In [ ]:
creds = read_credentials('vald_api_cred.txt')
client_id = creds['client_id']
client_secret = creds['client_secret']
tenant_id = creds['tenant_id']
print(f"Client ID: {client_id[:10]}...")
print(f"Tenant ID: {tenant_id}")

## Step 1: Authentication

In [ ]:
auth = ValdAuth(client_id, client_secret, tenant_id=tenant_id, region='USA')

### Get OAuth Access Token

In [ ]:
token = auth.get_token()
print(f"Token obtained: {token[:20]}...")

### Optional: Get Tenant Information

In [ ]:
all_tenants = auth.get_all_tenants()
print(f"Found {len(all_tenants)} tenant(s)")

In [ ]:
tenant_info = auth.get_tenant_info()
print(f"Tenant Name: {tenant_info.get('name')}")
print(f"Tenant ID: {tenant_info.get('id')}")

## Step 2: Get Categories and Groups

In [ ]:
categories_df = auth.get_tenant_categories()
print(categories_df[['name', 'id']].to_string(index=False))

In [ ]:
groups_df = auth.get_tenant_groups()
print(f"Found {len(groups_df)} group(s)")
print(groups_df[['name', 'id']].head(10).to_string(index=False))

## Step 3: Get Profiles

In [ ]:
group_name = 'Research'
category_name = 'Team'

try:
    profiles_df = auth.get_group_profiles(groupName=group_name, categoryName=category_name)
    print(f"Found {len(profiles_df)} profile(s)")
    print(profiles_df[['givenName', 'familyName', 'profileId']].head(10).to_string(index=False))
except Exception as e:
    print(f"Error: {e}")

## Step 4: Initialize NordBord API

In [ ]:
nb = NordBordAPI(tenant_id=auth.tenant_id, header=auth.headers, region='USA')

### Get Test Information

In [ ]:
date = '15/06/2026'
tests_df = nb.get_tests_info(date)

if tests_df is not None:
    print(f"Found {len(tests_df)} test(s)")
    print(tests_df[['testId', 'profileId']].head())
else:
    print("No tests found")

In [ ]:
if tests_df is not None and len(tests_df) > 0:
    test_id = tests_df.iloc[0]['testId']
    test = nb.get_test_results(test_id)
    if test is not None:
        print(test)
    else:
        print("No results available")

### Pull Nordbord fore trace for single test

In [ ]:
force_df = nb.get_force_trace(test_id)

In [ ]:
force_df

### Plot Data

In [ ]:
if force_df is not None and len(force_df) > 0:
    force_df.plot(x='time_s', y='leftForce', label='Left Force')
    force_df.plot(x='time_s', y='rightForce', label='Right Force')
    

In [ ]:
import matplotlib.pyplot as plt
print("Visualization complete")